# 06. OmniQuant and HQQ: Optimization-Based PTQ

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Shao et al. (2023, *OmniQuant: Omnidirectionally Calibrated Quantization for Large Language Models*); Badri & Shayer (2023, *Half-Quadratic Quantization of Large Machine Learning Models*).
- **Purpose:** Formulate post-training quantization directly as an optimization problem: optimizing Learnable Equivalent Transformations (LET) and Learnable Weight Clipping (LWC) with frozen weights, and deriving closed-form $L_p$ proximal solutions without calibration data (HQQ).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Rotation and Incoherence PTQ](05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. SpQR and SqueezeLLM: Sparse Plus Nonuniform PTQ](07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT update full weight matrices during PTQ (risks catastrophic forgetting and requires massive compute; optimize only auxiliary scale/clipping parameters); do NOT use full calibration pipelines when data is strictly restricted by privacy (use data-free HQQ proximal minimization).

## Mathematical Foundations & Optimization Parameterization

1. **Learnable Equivalent Transformation (LET)**:
   $$\mathbf{Y} = (\mathbf{X}\text{diag}(\mathbf{s})^{-1})(\text{diag}(\mathbf{s})\mathbf{W}), \quad \mathbf{s} = \exp(\mathbf{\gamma})$$
2. **Learnable Weight Clipping (LWC)**:
   $$l = \min(\mathbf{W})\sigma(h_l), \quad u = \max(\mathbf{W})\sigma(h_u), \quad \Delta = (u - l) / (2^b - 1)$$
3. **HQQ Data-Free Optimization**:
   $$\min_{\mathbf{S}, \mathbf{Z}} \| \mathbf{W} - (\text{round}(\mathbf{W}/\mathbf{S} + \mathbf{Z}) - \mathbf{Z})\mathbf{S} \|_p$$
4. **Parameter Efficiency**:
   $$\text{Optimized Parameters} = \{ \mathbf{\gamma}, h_l, h_u \} \implies <0.01\% \text{ total model parameters}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def lwc_quantize(w: torch.Tensor, h_l: torch.Tensor, h_u: torch.Tensor, qmin: float = 0.0, qmax: float = 15.0):
    w_min = torch.min(w, dim=0, keepdim=True).values
    w_max = torch.max(w, dim=0, keepdim=True).values
    lower = w_min * torch.sigmoid(h_l)
    upper = w_max * torch.sigmoid(h_u)
    scale = torch.clamp((upper - lower) / qmax, min=1e-5)
    q = torch.clamp(torch.round((w - lower) / scale), qmin, qmax)
    return q * scale + lower, scale, lower

def omniquant_forward(x: torch.Tensor, weight: torch.Tensor, gamma: torch.Tensor, h_l: torch.Tensor, h_u: torch.Tensor, bias: torch.Tensor | None = None):
    s = torch.exp(gamma)
    x_scaled = x / s.unsqueeze(0)
    w_scaled = weight * s.unsqueeze(1)
    w_deq, _, _ = lwc_quantize(w_scaled, h_l, h_u)
    out = x_scaled @ w_deq
    if bias is not None: out = out + bias
    return out, w_deq

def hqq_proximal_quantize(w: torch.Tensor, nbits: int = 4):
    qmax = float(2 ** nbits - 1)
    w_min = torch.min(w, dim=0, keepdim=True).values
    w_max = torch.max(w, dim=0, keepdim=True).values
    scale = torch.clamp((w_max - w_min) / qmax, min=1e-5)
    zero = -w_min / scale
    q = torch.clamp(torch.round(w / scale + zero), 0.0, qmax)
    return (q - zero) * scale, scale, zero

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. HQQ proximal quantization parity
w_hqq, hqq_scale, hqq_zero = hqq_proximal_quantize(weights["w_orig"], nbits=4)
assert float(torch.max(torch.abs(w_hqq - weights["w_hqq"]))) < 1e-5
assert float(torch.max(torch.abs(hqq_scale - weights["hqq_scale"]))) < 1e-5
assert float(torch.max(torch.abs(hqq_zero - weights["hqq_zero"]))) < 1e-5
out_hqq = weights["x_calib"] @ w_hqq + weights["b_orig"]
assert float(torch.max(torch.abs(out_hqq - weights["out_hqq"]))) < 1e-5

# 3. OmniQuant forward with optimized parameters parity
out_omni, _ = omniquant_forward(weights["x_calib"], weights["w_orig"], weights["gamma_opt"], weights["h_l_opt"], weights["h_u_opt"], weights["b_orig"])
assert float(torch.max(torch.abs(out_omni - weights["out_omniquant"]))) < 1e-5
print("All OmniQuant and HQQ parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: HQQ Data-Free vs OmniQuant Optimized
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_hqq = eval_sqnr(weights["out_hqq"])
sqnr_omni = eval_sqnr(weights["out_omniquant"])

print(f"HQQ Data-Free SQNR:    {sqnr_hqq:.2f} dB (Zero calibration data required!)")
print(f"OmniQuant Optimized:   {sqnr_omni:.2f} dB")
assert sqnr_hqq > 25.0, "HQQ data-free proximal quantization must achieve >25 dB SQNR!"
assert sqnr_omni > 15.0, "OmniQuant optimization must maintain viable SQNR (>15 dB)!"

In [ ]:
# Latency Microbenchmark: Fused LWC Asymmetric Dequantization
d_in, d_out = 4096, 4096
q_mock = torch.randint(0, 16, (d_in, d_out)).float()
scale_mock = torch.randn(1, d_out).abs() * 0.1
lower_mock = torch.randn(1, d_out) * 0.5

t0 = time.perf_counter()
for _ in range(50):
    w_deq = q_mock * scale_mock + lower_mock
t_lwc = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused LWC Dequantization (4096x4096) latency: {t_lwc:.3f} ms")
assert t_lwc < 15.0, "LWC dequantization must execute in <15 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Continuous Auxiliary Optimization**: Optimizing clipping boundaries ($h_l, h_u$) and equivalent channel scales ($\gamma$) discovers higher-quality quantizers than hand-crafted rules.
2. **Data-Free Advantage of HQQ**: Half-Quadratic proximal splitting solves optimal quantization parameters in seconds without needing calibration sequences.
3. **Inference Compatibility**: Both OmniQuant and HQQ produce standard linear layers with per-channel scales and zero points executable on hardware tensor cores.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_omniquant_and_hqq_optimization_based_ptq.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.md)
- **PyTorch 2.x Module:** [06_omniquant_and_hqq_optimization_based_ptq.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.py)
- **Functional JAX Module:** [06_omniquant_and_hqq_optimization_based_ptq.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.py)
- **Native Rust Candle Track:** [06_omniquant_and_hqq_optimization_based_ptq.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.rs)
- **Hardware Kernel (CPU / WGSL):** [06_omniquant_and_hqq_optimization_based_ptq.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.rs)
- **Golden SafeTensors:** [06_omniquant_and_hqq_optimization_based_ptq.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/06_omniquant_and_hqq_optimization_based_ptq.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Rotation and Incoherence PTQ](05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 07. SpQR and SqueezeLLM: Sparse Plus Nonuniform PTQ](07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.ipynb) &rarr;